### Complaint Agent — Evaluation

Standalone evaluation task for the deployed Complaint Agent App. Runs after
`Complaint_Triage_Agent` in the pipeline so the app is ready before we hit it.

- **Gated by `SKIP_EVAL`** (default `"true"`) — flip to `"false"` to actually
  run the evaluation. The default skip is a deliberate rate-limit safeguard:
  the complaint agent uses DSPy ReAct, which fires 5-10 LLM calls per invocation.
- Calls the deployed Databricks App via its MLflow AgentServer `/responses`
  contract rather than importing the agent module.
- Eval results land in `/Shared/{CATALOG}_complaint_agent_dev`.


In [ ]:
%pip install -U -qqqq mlflow-skinny[databricks] databricks-sdk requests
dbutils.library.restartPython()


In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

import os
import sys
sys.path.append(os.path.abspath("../utils"))
from agent_app_client import resolve_agent_app_name

try:
    _APP_NAME_PARAM = dbutils.widgets.get("COMPLAINT_AGENT_APP_NAME")
except Exception:
    _APP_NAME_PARAM = ""
APP_NAME = resolve_agent_app_name(_APP_NAME_PARAM, CATALOG, "complaint")

try:
    SKIP_EVAL = dbutils.widgets.get("SKIP_EVAL").strip().lower() == "true"
except Exception:
    SKIP_EVAL = True

import mlflow

DEV_EXPERIMENT = f"/Shared/{CATALOG}_complaint_agent_dev"
mlflow.set_experiment(DEV_EXPERIMENT)

print(f"Catalog:          {CATALOG}")
print(f"App:              {APP_NAME}")
print(f"SKIP_EVAL:        {SKIP_EVAL}")
print(f"Dev experiment:   {DEV_EXPERIMENT}")
print(f"MLflow version:   {mlflow.__version__}")


In [ ]:
if SKIP_EVAL:
    print(
        f"⏭  SKIP_EVAL=true — skipping mlflow.genai.evaluate to avoid the ~50 "
        f"LM-call eval burst (DSPy ReAct fires 5-10 LLM calls per agent invocation). "
        f"Pass --params \"SKIP_EVAL=false\" to actually run the evaluation."
    )
    dbutils.notebook.exit("skipped")

In [ ]:
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

POLL_INTERVAL_S = 15
MAX_POLLS = 40

print(f"Polling app readiness ({MAX_POLLS * POLL_INTERVAL_S // 60} min max)...")


def _app_state(a):
    cs = getattr(a, "compute_status", None)
    s = getattr(cs, "state", None) if cs is not None else None
    if s is None:
        s = getattr(a, "state", None)
    return getattr(s, "value", str(s)) if s is not None else ""


for attempt in range(1, MAX_POLLS + 1):
    try:
        app = w.apps.get(APP_NAME)
        state = _app_state(app)
        url = getattr(app, "url", "")
        if state in ("ACTIVE", "RUNNING", "READY") and url:
            print(f"  App ready: state={state}, url={url}")
            break
        print(f"  [{attempt}/{MAX_POLLS}] state={state or 'unknown'}, url={url or 'pending'}")
    except Exception as e:
        print(f"  poll error: {type(e).__name__}: {e}")
    time.sleep(POLL_INTERVAL_S)
else:
    raise RuntimeError(
        f"App {APP_NAME} did not become ready within "
        f"{MAX_POLLS * POLL_INTERVAL_S // 60} minutes."
    )


In [ ]:
import os
import random
import time

sys.path.append(os.path.abspath("../utils"))
from agent_app_client import call_agent_app_text

os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"
os.environ["MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS"] = "1"
os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "600")


def _is_rate_limit(exc: BaseException) -> bool:
    msg = str(exc)
    return (
        "REQUEST_LIMIT_EXCEEDED" in msg
        or "RateLimitError" in type(exc).__name__
        or "rate limit" in msg.lower()
    )


def _input_messages(value):
    if isinstance(value, dict):
        return value.get("input") or value.get("messages") or [value]
    return value


def predict_fn(input):
    """Call the deployed Complaint Agent App with retry-on-rate-limit."""
    max_attempts = 6
    input_messages = _input_messages(input)
    for attempt in range(max_attempts):
        try:
            return call_agent_app_text(
                app_name=APP_NAME,
                input_messages=input_messages,
                dbutils=dbutils,
                timeout=600,
            )
        except Exception as exc:
            if not _is_rate_limit(exc) or attempt == max_attempts - 1:
                raise
            backoff = min(60, 2 ** attempt) + random.uniform(0, 1)
            print(f"  rate-limited (attempt {attempt + 1}/{max_attempts}), sleeping {backoff:.1f}s")
            time.sleep(backoff)


print(f"predict_fn defined — will call app {APP_NAME}")


In [ ]:
import json as _json
import mlflow.genai.datasets

UC_DATASET_TABLE = f"{CATALOG}.evaluations.complaint_agent_eval_dataset"

# Read the dataset registered by stages/complaint_setup.ipynb. The setup task
# JSON-stringifies any list-typed field on write (Arrow + mlflow.genai.datasets
# can't reliably serialize ARRAY<STRING>), so we deserialize symmetrically.
_rows = spark.table(UC_DATASET_TABLE).toPandas().to_dict(orient="records")


def _deserialize_record(rec):
    """Recursively undo the JSON-stringification done by setup."""
    out = {}
    for k, v in rec.items():
        if isinstance(v, dict):
            out[k] = _deserialize_record(v)
        elif isinstance(v, str):
            try:
                parsed = _json.loads(v)
                out[k] = parsed if isinstance(parsed, (list, dict)) else v
            except (ValueError, TypeError):
                out[k] = v
        else:
            out[k] = v
    return out


EVAL_DATASET = [_deserialize_record(r) for r in _rows]

if not EVAL_DATASET:
    raise RuntimeError(
        f"{UC_DATASET_TABLE} is empty. Re-run the Complaint_Setup task first "
        "(it builds the dataset from {CATALOG}.lakeflow.all_events)."
    )

print(f"Loaded {len(EVAL_DATASET)} eval scenarios from {UC_DATASET_TABLE}")

# Complaint agent's prompt is a DSPy Signature compiled at runtime, NOT an
# entry in the MLflow Prompt Registry.  Tag eval runs with the signature
# class name so failed scorers point back at the right source.
PROMPT_MANAGEMENT = "dspy_signature"
PROMPT_MODULE = "ComplaintTriage"
print(f"  Complaint agent: {PROMPT_MANAGEMENT}={PROMPT_MODULE} (no Prompt Registry entry)")


In [ ]:
from mlflow.genai.scorers import Guidelines, Safety
try:
    from mlflow.genai.scorers import RelevanceToQuery
    _has_relevance = True
except ImportError:
    RelevanceToQuery = None
    _has_relevance = False
    print("⚠\ufe0f  RelevanceToQuery not available in this MLflow version — skipping it from the eval mix")

evidence_reasoning = Guidelines(
    name="evidence_reasoning",
    guidelines=[
        "The rationale should provide specific evidence or reasoning for the decision made",
        "For timing complaints, rationale should mention delivery times or timing data if available",
        "For missing item complaints, rationale should reference the items in question",
        "For escalations, rationale should explain why human judgment is needed",
    ],
)

credit_reasonableness = Guidelines(
    name="credit_reasonableness",
    guidelines=[
        "If decision='escalate', automatically pass this check",
        "If decision='suggest_credit' with credit_amount > $0, amount should be reasonable ($5-$50)",
        "A credit_amount of $0 is valid when rationale indicates no issue",
    ],
)

decision_metadata = Guidelines(
    name="decision_metadata",
    guidelines=[
        "If decision='suggest_credit': confidence should be present and priority should be null",
        "If decision='escalate': priority should be present and confidence should be null",
    ],
)

_DATASET_SCORERS = [
    evidence_reasoning,
    credit_reasonableness,
    decision_metadata,
    Safety(),
]
if _has_relevance:
    _DATASET_SCORERS.append(RelevanceToQuery())

print(f"✅ Scorers defined ({len(_DATASET_SCORERS)}): " + ", ".join(s.name if hasattr(s, 'name') else type(s).__name__ for s in _DATASET_SCORERS))

In [ ]:
import mlflow
import mlflow.genai

_mlflow_dataset = mlflow.data.from_spark(
    spark.table(UC_DATASET_TABLE),
    table_name=UC_DATASET_TABLE,
)

with mlflow.start_run(run_name=f"{CATALOG}-complaint-agent-eval"):
    mlflow.log_input(_mlflow_dataset, context="eval")
    mlflow.log_param("prompt_management", PROMPT_MANAGEMENT)
    mlflow.log_param("prompt_module", PROMPT_MODULE)
    results = mlflow.genai.evaluate(
        data=EVAL_DATASET,
        scorers=_DATASET_SCORERS,
        predict_fn=predict_fn,
    )

print("\n✅ Evaluation complete")
print(f"  metrics: {getattr(results, 'metrics', 'N/A')}")

## Trace-derived eval — recent production calls

The block above runs curated complaint scenarios. That's good for catching
regressions on **known-good cases**, but it doesn't tell us how the agent
is doing on **real production traffic**.

This section harvests the most recent successful traces from the prod
experiment (`/Shared/{CATALOG}_complaint_agent_prod` — populated whenever
the streaming pipeline triggers a complaint), turns them back into eval
inputs, and re-runs the agent against them with global quality guidelines.

This mirrors the pattern in `demos/operational-dashboard-demo/evaluation.ipynb`
for the Operational Supervisor, so all three agents share the same flywheel:

```
Production traffic → MLflow traces → resampled eval dataset → quality scoring
```

Skipped gracefully if the prod experiment doesn't exist yet or has no
traces (fresh deploy with no traffic).

In [ ]:
import mlflow
import pandas as pd

prod_experiment_name = f"/Shared/{CATALOG}_complaint_agent_prod"
trace_eval_data = []
traces_df = pd.DataFrame()

try:
    _exp = mlflow.get_experiment_by_name(prod_experiment_name)
except Exception as exc:
    print(f"  Could not look up prod experiment: {type(exc).__name__}: {exc}")
    _exp = None

if _exp is None:
    print(
        f"Skipping trace eval — experiment {prod_experiment_name} not found.\n"
        "  Send some complaints through the deployed app first."
    )
else:
    print(f"Found prod traces experiment: {prod_experiment_name} ({_exp.experiment_id})")

    # `return_type="list"` is required — `mlflow.search_traces()` returns a
    # pandas DataFrame by default in MLflow 3, and iterating it yields column
    # names (strings).  The iteration below expects `Trace` objects with
    # `.data.spans` / `.info.request_id`, so we explicitly request the list form.
    traces = mlflow.search_traces(
        experiment_ids=[_exp.experiment_id],
        filter_string="status = 'OK'",
        max_results=50,
        order_by=["timestamp DESC"],
        return_type="list",
    )
    print(f"Pulled {len(traces)} OK-status traces (most recent 50).")

    records = []
    for trace in traces:
        try:
            root = trace.data.spans[0]
            inputs_raw = root.inputs or {}
            # ResponsesAgent's request shape is `{"input": [{"role": ..., "content": ...}]}`,
            # but defensively also accept `messages` in case the SDK ever normalises it.
            items = inputs_raw.get("input") or inputs_raw.get("messages") or []
            complaint = next(
                (m["content"] for m in items if m.get("role") == "user"),
                None,
            )
            if complaint:
                records.append({
                    "trace_id": trace.info.request_id,
                    "timestamp_ms": trace.info.timestamp_ms,
                    "latency_ms": trace.info.execution_time_ms,
                    "complaint": complaint,
                })
        except Exception as _te:
            print(f"  ⚠️  skipped trace {getattr(trace.info, 'request_id', '?')}: {_te}")
            continue

    traces_df = pd.DataFrame(records)
    print(f"Extracted {len(traces_df)} usable user-complaint records.")

    if not traces_df.empty:
        TRACE_EVAL_LIMIT = 20
        trace_eval_data = [
            {"inputs": {"input": [{"role": "user", "content": row["complaint"]}]}}
            for _, row in traces_df.head(TRACE_EVAL_LIMIT).iterrows()
        ]
        print(f"Trace-derived eval dataset: {len(trace_eval_data)} records (cap={TRACE_EVAL_LIMIT}).")
        display(traces_df[["complaint", "latency_ms"]].head(10))

In [ ]:
from mlflow.genai.scorers import Guidelines, Safety

_TRACE_GUIDELINE = (
    "Response must clearly state a decision (suggest_credit or escalate) and "
    "the rationale must reference concrete evidence from the complaint. "
    "Response must not be a generic hedge or refusal. "
    "Response must directly address the complaint."
)

if trace_eval_data:
    _trace_df = pd.DataFrame(
        [{"complaint": row["inputs"]["input"][0]["content"]} for row in trace_eval_data]
    )
    _trace_dataset = mlflow.data.from_pandas(
        _trace_df,
        source=f"{prod_experiment_name} (last {len(trace_eval_data)} OK traces)",
        name="complaint_agent_trace_eval_dataset",
    )

    with mlflow.start_run(run_name=f"{CATALOG}-complaint-agent-trace-eval"):
        mlflow.log_input(_trace_dataset, context="trace_eval")
        mlflow.log_param("prompt_management", PROMPT_MANAGEMENT)
        mlflow.log_param("prompt_module", PROMPT_MODULE)
        trace_results = mlflow.genai.evaluate(
            data=trace_eval_data,
            scorers=[
                evidence_reasoning,
                credit_reasonableness,
                decision_metadata,
                Guidelines(name="trace_quality", guidelines=_TRACE_GUIDELINE),
                Safety(),
                *([RelevanceToQuery()] if _has_relevance else []),
            ],
            predict_fn=predict_fn,
        )
    print("\n✅ Trace-derived evaluation complete")
    print(f"  metrics: {getattr(trace_results, 'metrics', 'N/A')}")
else:
    print("Skipping trace evaluation — no trace data available yet.")